In [8]:
import pandas as pd
import os
from functools import reduce

DATASET_FOLDER = "datasets/raw"

files = [
    "TamilNadu_Rainfall_2015_2023.csv",
    "TamilNadu_Temperature_2015_2023.csv",
    "TamilNadu_Maximum_Temperature_2015_2023.csv",
    "TamilNadu_Minimum_Temperature_2015_2023.csv",
    "TamilNadu_Humidity_2015_2023.csv",
    "TamilNadu_NDVI_2015_2023.csv",
    "TamilNadu_EVI_2015_2023.csv",
    "TamilNadu_LST_2015_2023.csv",
    "TamilNadu_Soil_Moisture_2015_2023.csv",
    "TamilNadu_ET_2015_2023.csv",
    "TamilNadu_Solar_Radiation_2015_2023.csv"
]

static_files = [
    "TamilNadu_Soil_Organic_Carbon.csv",
    "TamilNadu_Soil_pH.csv"
]

dynamic_dfs = []

print("=" * 70)
print("Reading Dynamic Datasets")
print("=" * 70)

# -------------------------------
# Dynamic Features
# -------------------------------
for file in files:

    path = os.path.join(DATASET_FOLDER, file)

    df = pd.read_csv(path)

    feature = [c for c in df.columns
               if c not in ["system:index", "District", "Year", ".geo"]][0]

    print(f"{file}  -->  {feature}")

    df = df[["District", "Year", feature]]

    df["District"] = df["District"].astype(str).str.strip()
    df["Year"] = df["Year"].astype(int)

    dynamic_dfs.append(df)

# Merge dynamic datasets
environment = reduce(
    lambda left, right:
    pd.merge(left, right, on=["District", "Year"], how="outer"),
    dynamic_dfs
)

print("\nDynamic Merge Shape:", environment.shape)

print("\nReading Static Datasets")

# -------------------------------
# Static Features
# -------------------------------
for file in static_files:

    path = os.path.join(DATASET_FOLDER, file)

    df = pd.read_csv(path)

    feature = [c for c in df.columns
               if c not in ["system:index", "District", ".geo"]][0]

    print(f"{file}  -->  {feature}")

    df = df[["District", feature]]

    df["District"] = df["District"].astype(str).str.strip()

    environment = environment.merge(
        df,
        on="District",
        how="left"
    )

# -------------------------------
# Sort
# -------------------------------
environment = environment.sort_values(
    ["District", "Year"]
).reset_index(drop=True)

# -------------------------------
# Save
# -------------------------------
environment.to_csv(
    "Environmental_Features.csv",
    index=False
)

print("\n" + "=" * 70)
print("SUCCESS")
print("=" * 70)

print(environment.head())

print("\nShape:", environment.shape)

print("\nColumns:")
print(environment.columns.tolist())

print("\nDistricts:", environment["District"].nunique())
print("Years:", environment["Year"].nunique())

Reading Dynamic Datasets
TamilNadu_Rainfall_2015_2023.csv  -->  Rainfall_mm
TamilNadu_Temperature_2015_2023.csv  -->  Avg_Temperature_C
TamilNadu_Maximum_Temperature_2015_2023.csv  -->  Max_Temperature_C
TamilNadu_Minimum_Temperature_2015_2023.csv  -->  Min_Temperature_C
TamilNadu_Humidity_2015_2023.csv  -->  Relative_Humidity_Percent
TamilNadu_NDVI_2015_2023.csv  -->  NDVI
TamilNadu_EVI_2015_2023.csv  -->  EVI
TamilNadu_LST_2015_2023.csv  -->  LST_C
TamilNadu_Soil_Moisture_2015_2023.csv  -->  Soil_Moisture
TamilNadu_ET_2015_2023.csv  -->  Evapotranspiration_mm
TamilNadu_Solar_Radiation_2015_2023.csv  -->  Solar_Radiation_MJ

Dynamic Merge Shape: (342, 13)

Reading Static Datasets
TamilNadu_Soil_Organic_Carbon.csv  -->  Soil_Organic_Carbon
TamilNadu_Soil_pH.csv  -->  Soil_pH

SUCCESS
   District  Year  Rainfall_mm  Avg_Temperature_C  Max_Temperature_C  \
0  Ariyalur  2015  1547.374510          27.965393          35.400772   
1  Ariyalur  2016   891.978852          28.610633          36

In [7]:
import pandas as pd

df = pd.read_csv("datasets/raw/TamilNadu_Soil_pH.csv")

print(df.columns.tolist())
print(df.head())

['system:index', 'District', 'Soil_pH', '.geo']
           system:index    District    Soil_pH  \
0  00000000000000000000    Ariyalur  65.126477   
1  00000000000000000002     Chennai  60.169581   
2  00000000000000000003  Coimbatore  64.395477   
3  00000000000000000004   Cuddalore  63.108627   
4  00000000000000000005  Dharmapuri  65.152916   

                                     .geo  
0  {"type":"MultiPoint","coordinates":[]}  
1  {"type":"MultiPoint","coordinates":[]}  
2  {"type":"MultiPoint","coordinates":[]}  
3  {"type":"MultiPoint","coordinates":[]}  
4  {"type":"MultiPoint","coordinates":[]}  
